In [24]:
import pandas as pd
import os
from dotenv import load_dotenv
from googleapiclient.discovery import build

load_dotenv()
API_KEY = os.getenv("YOUTUBE_API_KEY")

print("API Key loaded:", "✅ Yes" if API_KEY else "❌ Not found")

API Key loaded: ✅ Yes


In [27]:
import pandas as pd
import numpy as np

# Load the already-saved enriched data
df_final = pd.read_csv("../02_Datasets/processed/youtube_influencers_enriched.csv")
print(f"Loaded: {df_final.shape}")
print(df_final.columns.tolist())

Loaded: (5340, 21)
['channel_name', 'channel_id', 'subscribers', 'total_views', 'video_count', 'avg_views_per_video', 'view_per_subscriber', 'country', 'tier', 'niche', 'api_subscribers', 'api_total_views', 'api_video_count', 'api_comment_count', 'uploads_playlist', 'avg_video_views', 'avg_video_likes', 'avg_video_comments', 'engagement_rate', 'like_rate', 'comment_rate']


In [29]:
from googleapiclient.discovery import build
import time

youtube = build("youtube", "v3", developerKey=API_KEY)

def get_last_upload_date(playlist_id):
    try:
        response = youtube.playlistItems().list(
            part="contentDetails",
            playlistId=playlist_id,
            maxResults=1
        ).execute()
        items = response.get("items", [])
        if items:
            return items[0]["contentDetails"].get("videoPublishedAt", None)
        return None
    except:
        return None

last_upload_data = {}
skipped = 0

print("Fetching last upload dates...")
for idx, row in df_final.iterrows():
    playlist_id = row.get("uploads_playlist")
    if not playlist_id or pd.isna(playlist_id):
        skipped += 1
        continue
    last_upload_data[row["channel_id"]] = get_last_upload_date(playlist_id)
    time.sleep(0.05)
    if idx % 500 == 0:
        print(f"  Progress: {idx} / {len(df_final)}")

print(f"\n✅ Done. Dates fetched for {len(last_upload_data)} channels ({skipped} skipped)")

Fetching last upload dates...
  Progress: 0 / 5340
  Progress: 500 / 5340
  Progress: 1000 / 5340
  Progress: 1500 / 5340
  Progress: 2000 / 5340
  Progress: 2500 / 5340
  Progress: 3000 / 5340
  Progress: 3500 / 5340
  Progress: 4000 / 5340
  Progress: 4500 / 5340
  Progress: 5000 / 5340

✅ Done. Dates fetched for 5330 channels (10 skipped)


In [30]:
upload_df = pd.DataFrame.from_dict(last_upload_data, orient="index").reset_index()
upload_df.columns = ["channel_id", "last_upload_date"]
upload_df["last_upload_date"] = pd.to_datetime(upload_df["last_upload_date"], utc=True)

df_final = df_final.merge(upload_df, on="channel_id", how="left")

now = pd.Timestamp.now(tz="UTC")
df_final["days_since_last_upload"] = (now - df_final["last_upload_date"]).dt.days

print("✅ Date columns added")
print(f"Shape: {df_final.shape}")
print(f"\nDays since last upload — summary:")
print(df_final["days_since_last_upload"].describe().round(1))
print(f"\nChannels with missing date: {df_final['days_since_last_upload'].isna().sum()}")

✅ Date columns added
Shape: (5340, 23)

Days since last upload — summary:
count    5323.0
mean      211.5
std       537.3
min         0.0
25%         1.0
50%         6.0
75%       115.0
max      6160.0
Name: days_since_last_upload, dtype: float64

Channels with missing date: 17


In [31]:
df_clean = df_final[df_final["days_since_last_upload"] <= 365].copy()

print(f"Before filter: {len(df_final)} rows")
print(f"After ≤365 days: {len(df_clean)} rows")
print(f"Removed: {len(df_final) - len(df_clean)} inactive channels")

output_path = "../02_Datasets/processed/youtube_influencers_enriched.csv"
df_clean.to_csv(output_path, index=False)

print(f"\n✅ Final dataset saved")
print(f"   Rows: {len(df_clean)} | Columns: {len(df_clean.columns)}")
print(f"   Filters applied:")
print(f"   ✅ subscribers ≥ 100")
print(f"   ✅ api_video_count ≥ 5")
print(f"   ✅ days_since_last_upload ≤ 365")

Before filter: 5340 rows
After ≤365 days: 4527 rows
Removed: 813 inactive channels

✅ Final dataset saved
   Rows: 4527 | Columns: 23
   Filters applied:
   ✅ subscribers ≥ 100
   ✅ api_video_count ≥ 5
   ✅ days_since_last_upload ≤ 365


## Data Quality Check — Enriched Dataset

Before using this dataset for analysis, we identified and removed records 
that do not represent real, active YouTube creators.

### Issues Found:
| Issue | Count | Reason to Remove |
|-------|-------|-----------------|
| Channels with < 100 subscribers | 1,867 | Spam/ghost accounts, not real influencers |
| Channels with 0 total views | 74 | Dead/inactive channels |
| Channels with 0 VPS | 6 | Mathematically invalid |
| **Total removed** | **1,905** | |

### Impact on Key Metric (Nano VPS):
- Without filter: nano median VPS = **4.82** ← inflated by spam channels
- With filter: nano median VPS = **1.90** ← correct, consistent with H1 analysis

This filter matches the one applied in all hypothesis testing notebooks.

## How We Found These Issues

Standard EDA checks run on the enriched dataset revealed three red flags:

---
## Final Dataset Filters

Three filters define an "active, genuine YouTube creator" for this analysis.

| Filter | Threshold | Reason |
|--------|-----------|--------|
| Subscribers | ≥ 100 | Removes ghost/spam accounts |
| Video count | ≥ 5 | Confirms real channel presence |
| Last upload | ≤ 365 days | Confirms currently active creator |

**What we deliberately excluded and why:**
- ❌ Channel creation date — misleading. A 2009 channel posting last month is more relevant than a 2023 channel that went silent. Activity proves presence, creation date does not.
- ❌ 30-day cutoff — biases against nano creators who post less frequently but are genuine. Median days since upload = 79 days confirms most channels are active.